# 🌿 BioEcoInt — Pipeline de Treinamento Local & SOTA Benchmark (Workstation GPU)
> **Laboratório IFMaker / BioEcoInt** — *Engenharia de Software & Machine Learning Aplicada à Fauna Amazônica*
> 
> Este notebook executa localmente nesta estação de trabalho a **Bateria Completa de Benchmarks Multi-Etapas**:
> 1. **Fase 1 (Arquiteturas Mobile)**: ResNet18 Baseline, Two-Stage YOLO Crop, ResNet18 Coordinate Attention, MobileNetV4-Conv-Small e RepViT-M0.9.
> 2. **Fase 2 (Funções de Perda)**: Cross-Entropy com Label Smoothing, Focal Loss (gamma=2.0) e Balanced Softmax / Logit Adjustment (tau=1.0).
> 3. **Fase 3 & 4 (Paradigma P3 - SAM 2 Tap-to-Crop)**: Classificador treinado sobre a Região de Interesse (ROI) recortada de alta precisão pelo Segment Anything 2.
> 4. **Fase 5 (Avaliação em Vídeo & Comitê Temporal)**: Simulação do uso em campo no app Flutter (janela de 2s @ 3 FPS, regra de consenso >= 70%).
> 5. **Fase 6 (Otimização HPO)**: Optuna TPE com poda assíncrona ASHA (Successive Halving).
> 6. **Fase 7 (Deploy Mobile & Quantização)**: Exportação ONNX, TFLite Float16 e TFLite INT8 PTQ.
> 
> ⚡ **Vantagens da Execução Local**:
> - Uso direto do hardware local (RTX 2060 SUPER 8GB VRAM + 20 vCPUs + 32GB RAM);
> - Leitura instantânea no SSD NVMe sem latência de FUSE/Google Drive;
> - Zero risco de timeout de sessão ou consumo de Compute Units (CUs) do Colab.


### 1. Diagnóstico do Hardware Local (GPU RTX 2060 SUPER)
Verifica a disponibilidade da GPU dedicada e memória de vídeo (VRAM) disponível.

In [ ]:
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated_vram = torch.cuda.memory_allocated(0) / (1024**2)
    print(f"VRAM Total: {total_vram:.2f} GB | VRAM Alocada: {allocated_vram:.2f} MB")
!nvidia-smi


### 2. Sincronização do Ambiente Virtual Local (`uv sync`)
Garante que todas as bibliotecas (`torch`, `torchvision`, `ultralytics`, `timm`, `optuna`) estão sincronizadas com o `pyproject.toml`.

In [ ]:
!uv sync
!uv run python -c "import torch, torchvision, ultralytics, timm; print('✓ Ambiente local verificado! PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"


### 3. Verificação dos Datasets Locais (SSD NVMe)
Confirma que os dois datasets principais estão prontos para treinamento no diretório local `datasets/generated/`.

In [ ]:
import os
from pathlib import Path

datasets_root = Path("datasets/generated")
sanitized_dir = datasets_root / "animals_video_sanitized"
sam2_dir = datasets_root / "animals_video_sam2_crops"

print(f"Dataset 1 (Sanitizado Full/Center-Crop): {'✓ Presente' if sanitized_dir.exists() else '❌ Ausente'}")
print(f"Dataset 2 (SAM 2 Tap-to-Crop):          {'✓ Presente' if sam2_dir.exists() else '❌ Ausente'}")

# Descompactação automática caso apenas o zip esteja presente no disco
if not sanitized_dir.exists() and (datasets_root / "animals_video_sanitized.zip").exists():
    print("Descompactando animals_video_sanitized.zip...")
    !unzip -q -o datasets/generated/animals_video_sanitized.zip -d datasets/generated/

if not sam2_dir.exists() and (datasets_root / "animals_video_sam2_crops.zip").exists():
    print("Descompactando animals_video_sam2_crops.zip...")
    !unzip -q -o datasets/generated/animals_video_sam2_crops.zip -d datasets/generated/

!ls -lh datasets/generated/


### 4. Bateria de Smoke Tests & Verificação de Vazamento (Pré-Voo)
Executa uma validação rápida de forward/backward em todos os modelos e confirma **zero data leakage** nos manifestos.

In [ ]:
!uv run python tests/smoke_test_all_experiments.py
!uv run python tests/test_leakage_detector.py


### 5. 🚀 Experimento 1: ResNet18 Baseline Sanado
- **Arquitetura**: ResNet18 pré-treinada com Two-Stage Transfer Learning.
- **Divisão**: Vídeos estritamente disjuntos por classe (Zero Shuffle Leakage).
- **Otimizador**: AdamW com taxas de aprendizado diferenciais e RandAugment nativo na GPU.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp1_resnet18_baseline.yml


### 6. 🎯 Experimento 2 (SOTA 1): Two-Stage YOLO Crop Classifier
- **Arquitetura**: Detector YOLOv8n localiza a ROI da fauna (+15% padding) -> ResNet18 classifica exclusivamente o animal.
- **Fix Aplicado**: YOLO mantido estritamente fora do `_modules` do PyTorch para eliminar vazamento no Model EMA e descompasso de device.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp2_resnet18_yolo_crop.yml


### 7. 🧠 Experimento 3 (SOTA 2): ResNet18 com Coordinate Attention (CA)
- **Arquitetura**: ResNet18 integrando blocos de Atenção Coordenada 2D (Hou et al., CVPR) nos estágios Layer3 e Layer4.
- **Benefício**: Codifica relações espaciais de longo alcance sem o custo quadrático de Vision Transformers.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp3_resnet18_ca.yml


### 8. ⚡ Experimento 4 (SOTA 3): Google MobileNetV4-Conv-Small com Balanced Softmax & EMA
- **Arquitetura**: `mobilenetv4_conv_small.e2400_r224_in1k` (3.8M parâmetros, Google 2024).
- **Função de Perda**: **Balanced Softmax / Logit Adjustment** com tau=1.0 para mitigar o desbalanceamento de espécies raras.
- **Estabilidade**: Model EMA (beta=0.999), Differential LR e Weight Decay Seletivo.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_small.yml


### 9. 🚀 Experimento 5 (SOTA 4): RepViT-M0.9 com Balanced Softmax & EMA
- **Arquitetura**: `repvit_m0_9` (4.7M parâmetros) com reparametrização estrutural no deploy mobile.
- **Otimização**: Balanced Softmax (tau=1.0), Model EMA e Two-Stage Warmup.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9.yml


### 10. 🔬 Experimento 6 (Fase 2): Benchmark de Funções de Perda (Iso-condicional na MobileNetV4)
Compara sob a mesma arquitetura MobileNetV4-Conv-Small:
1. **Cross-Entropy Pura** com Label Smoothing (0.1);
2. **Focal Loss** (gamma=2.0) para alvos camuflados e desafiadores;
3. *(O experimento 4 acima serviu como a condição com Balanced Softmax tau=1.0)*.

In [ ]:
# 1. Treino com Cross-Entropy pura + Label Smoothing 0.1
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_ce.yml

# 2. Treino com Focal Loss (gamma=2.0) + Label Smoothing 0.1
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_focal.yml


### 11. 🎯 Experimento 7 (Fase 4): Paradigma P3 — Treinamento sobre ROI Recortada via SAM 2
- **Conceito**: Treina a MobileNetV4 diretamente sobre os recortes da fauna isolados pelo SAM 2 (+15% margem dinâmica).
- **Vantagem**: Reduz ruídos de fundo e foca nas texturas e contornos tridimensionais das esculturas.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_sam2_crops.yml


### 12. 📹 Experimento 8 (Fase 5): Benchmark de Avaliação em Vídeo & Comitê Temporal (2s @ 3 FPS)
Simula a rotina em campo no app Flutter: amostra 6 frames em 2 segundos e calcula a acurácia de consenso temporal (>= 70% dos votos) contra a predição isolada de quadro único.

In [ ]:
# 1. Comitê Temporal no modelo MobileNetV4 Direto
!uv run base-test-committee \
  -opt options/exp_mobilenetv4_small.yml \
  -c experiments/Exp_MobileNetV4_Small_BalancedSoftmax/models/best_macro_f1.pth \
  --threshold 0.70 \
  --fps 3.0 \
  --duration 2.0 \
  --output experiments/Exp_MobileNetV4_Small_BalancedSoftmax/committee_direct_report.json

# 2. Comitê Temporal no modelo treinado sobre recortes SAM 2
!uv run base-test-committee \
  -opt options/exp_mobilenetv4_sam2_crops.yml \
  -c experiments/Exp_MobileNetV4_SAM2_Crops/models/best_macro_f1.pth \
  --use-sam2 \
  --threshold 0.70 \
  --fps 3.0 \
  --duration 2.0 \
  --output experiments/Exp_MobileNetV4_SAM2_Crops/committee_sam2_report.json


### 13. ⚙️ Experimento 9 (Fase 6): Otimização de Hiperparâmetros HPO (Optuna + ASHA Pruning)
Busca bayesiana TPE com poda agressiva por Sucessivas Reduções (ASHA) no modelo campeão.

In [ ]:
!uv run python experiments/hpo_optuna_baseline.py \
  -opt options/exp_mobilenetv4_small.yml \
  --n-trials 15 \
  --epochs-per-trial 5


### 14. 📱 Exportação Mobile e Quantização (TFLite FP16 & INT8 PTQ)
Exporta os modelos campeões para ONNX, TFLite Float16 e TFLite INT8 completo.

In [ ]:
!MPLBACKEND=Agg uv run base-export -opt options/exp1_resnet18_baseline.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_mobilenetv4_small.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_mobilenetv4_sam2_crops.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_repvit_m0_9.yml


### 15. 📊 Comparativo Consolidado de Métricas de Todos os Experimentos
Gera tabela pandas comparativa com Macro-F1, Acurácia balanceada e relatórios do comitê temporal.

In [ ]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

exp_dirs = [
    "experiments/Exp1_ResNet18_Baseline_Sanitized",
    "experiments/Exp2_ResNet18_YOLO_Crop",
    "experiments/Exp3_ResNet18_CoordinateAttention",
    "experiments/Exp_MobileNetV4_Small_BalancedSoftmax",
    "experiments/Exp_RepViT_M0_9_BalancedSoftmax",
    "experiments/Exp_Loss_CE_MobileNetV4",
    "experiments/Exp_Loss_Focal_MobileNetV4",
    "experiments/Exp_MobileNetV4_SAM2_Crops",
]

results = []
for d in exp_dirs:
    metrics_file = os.path.join(d, "val_metrics.json")
    log_files = glob.glob(os.path.join(d, "*.log"))
    exp_name = os.path.basename(d)
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            data = json.load(f)
            results.append({"Experimento": exp_name, **data})
    elif log_files:
        results.append({"Experimento": exp_name, "Status": "Concluído (Ver log)"})

if results:
    df = pd.DataFrame(results)
    display(df)
else:
    print("Nenhum arquivo de métricas encontrado ainda.")

# Exibir relatórios do Comitê Temporal se disponíveis
for rep_path in ["experiments/committee_mobilenetv4_direct.json", "experiments/committee_mobilenetv4_sam2.json"]:
    if os.path.exists(rep_path):
        with open(rep_path) as f:
            rep_data = json.load(f)
        print(f"\n=== Relatório do Comitê: {rep_path} ===")
        print("Single-Frame Acc:", rep_data.get("single_frame_metrics", {}).get("accuracy"))
        print("Committee Acc (decided):", rep_data.get("committee_metrics", {}).get("accuracy_on_decided"))
        print("Abstention Rate:", rep_data.get("committee_metrics", {}).get("abstention_rate"))


### 16. 🔔 Notificação de Conclusão na Área de Trabalho
Dispara uma notificação nativa no sistema operacional avisando que os treinamentos terminaram.

In [ ]:
import os
try:
    os.system('notify-send "BioEcoInt Lab" "Bateria de Treinamentos e Benchmarks Locais Concluída com Sucesso!" --icon=dialog-information')
    print("🔔 Notificação enviada para a área de trabalho!")
except Exception as e:
    print(f"Aviso ao emitir notificação: {e}")
